# Ingeniería de datos con Databricks - Construyendo nuestra plataforma de IoT de manufactura

Construir una plataforma de IoT requiere ingerir múltiples fuentes de datos.  

Es un proceso complejo que requiere cargas por lotes e ingesta en streaming para soportar insights en tiempo real, utilizados para monitoreo en tiempo real, entre otros.

Ingerir, transformar y limpiar datos para crear tablas SQL limpias para nuestros usuarios downstream (Analistas de Datos y Científicos de Datos) es complejo.

<link href="https://fonts.googleapis.com/css?family=DM Sans" rel="stylesheet"/>
<div style="width:300px; text-align: center; float: right; margin: 30px 60px 10px 10px;  font-family: 'DM Sans'">
  <div style="height: 300px; width: 300px;  display: table-cell; vertical-align: middle; border-radius: 50%; border: 25px solid #fcba33ff;">
    <div style="font-size: 70px;  color: #70c4ab; font-weight: bold">
      73%
    </div>
    <div style="color: #1b5162;padding: 0px 30px 0px 30px;">de los datos empresariales no se utilizan para analítica y toma de decisiones</div>
  </div>
  <div style="color: #bfbfbf; padding-top: 5px">Fuente: Forrester</div>
</div>

<br>

## <img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/john.png" style="float:left; margin: -35px 0px 0px 0px" width="80px"> John, como ingeniero de datos, invierte muchísimo tiempo…


* Programar a mano la ingesta y transformaciones de datos y lidiar con retos técnicos:<br>
  *Soportar streaming y batch, manejar operaciones concurrentes, problemas de archivos pequeños, requisitos GDPR, dependencias de DAG complejos...*<br><br>
* Construir frameworks personalizados para hacer cumplir calidad y pruebas<br><br>
* Construir y mantener infraestructura escalable, con observabilidad y monitoreo<br><br>
* Gestionar modelos de gobierno incompatibles de distintos sistemas
<br style="clear: both">

Esto resulta en **complejidad operativa** y sobrecarga, requiere perfiles expertos y, en última instancia, **pone en riesgo los proyectos de datos**.


<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=984752964297111&notebook=%2F01-Data-ingestion%2F01.2-SDP-python%2F01.1-SDP-Wind-Turbine-Python&demo_name=lakehouse-iot-platform&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-iot-platform%2F01-Data-ingestion%2F01.2-SDP-python%2F01.1-SDP-Wind-Turbine-Python&version=1&user_hash=53e7df68e5fee236d97fc15226aeeed74331d6f7c2f836f9b915aecc5654a25a">

# Simplifica la ingesta y la transformación con Spark Declarative Pipelines

<img style="float: right" width="500px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/refs/heads/main/images/manufacturing/lakehouse-iot-turbine/team_flow_john.png" />

En este notebook, trabajaremos como Ingeniero de Datos para construir nuestra plataforma de IoT. <br>
Ingeriremos y limpiaremos nuestras fuentes de datos en bruto para preparar las tablas requeridas por nuestras cargas de BI y ML.

Databricks simplifica esta tarea con Spark Declarative Pipelines al hacer la ingeniería de datos accesible para todos.

Spark Declarative Pipelines permite a los analistas de datos crear pipelines avanzados con SQL puro o Python.

¡Tu Spark Declarative Pipeline ha sido instalado e iniciado por ti! Abre el <a dbdemos-pipeline-id="sdp-sql" href="#joblist/pipelines/4d2b04f6-270a-49da-9980-96f393f6ef8c" target="_blank">Spark Declarative Pipeline de turbinas eólicas IoT</a> para verlo en acción.<br/>

*(Nota: El pipeline se iniciará automáticamente una vez que el job de inicialización se complete con dbdemos; esto puede tardar unos minutos... Revisa los logs de instalación para más detalles)*

## Construyendo un Spark Declarative Pipeline para ingerir sensores IoT y detectar equipos defectuosos

En este ejemplo, implementaremos un pipeline de extremo a extremo con Spark Declarative Pipelines consumiendo nuestros datos de sensores de turbinas eólicas. <br/>
Usaremos la arquitectura medallion, pero podríamos construir un esquema en estrella, data vault u otro modelo.

Cargaremos incrementalmente nuevos datos con Auto Loader, enriqueceremos esta información y luego cargaremos un modelo desde MLflow para realizar nuestro análisis de mantenimiento predictivo.

Esta información se utilizará para construir nuestros tableros AI/BI y hacer seguimiento del estado del parque eólico, el impacto de equipos defectuosos y recomendaciones para reducir el tiempo de inactividad potencial.

### Conjunto de datos:

* <strong>Metadatos de turbina</strong>: ID de turbina, ubicación (1 fila por turbina)
* <strong>Flujo de sensores de turbina</strong>: Flujo de streaming en tiempo real desde sensores (vibración, energía producida, velocidad, etc.)
* <strong>Estado de la turbina</strong>: Estado histórico de la turbina para analizar qué parte está defectuosa (usado como etiqueta en nuestro modelo de ML)

Implementemos el siguiente flujo: 
 
<div><img width="1100px" src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-full.png"/></div>

*Nota: estamos incluyendo el modelo de ML que nuestro [Científico de Datos construyó]($../04-Data-Science-ML/04.1-automl-predictive-maintenance-turbine) usando Databricks AutoML para mantenimiento predictivo. Lo cubriremos en la siguiente sección.*


## 1/ Exploración de datos

Todos los proyectos de datos comienzan con algo de exploración. Abre el cuaderno [/explorations/sample_exploration]($./explorations/sample_exploration) para comenzar y descubrir los datos disponibles para ti


## 2/ Ingesta de datos: capa Bronze

<div><img src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-1.png" width="700px" style="float: right"/></div>

Ingerir datos desde fuentes de streaming puede ser desafiante. En este ejemplo cargaremos incrementalmente los archivos desde nuestro almacenamiento en la nube, obteniendo solo los nuevos (en cuasi tiempo real o lanzado cada X horas).

Ten en cuenta que, aunque nuestros datos de streaming se agregan a nuestro almacenamiento en la nube, podríamos ingerir desde Kafka directamente: `.format(kafka)`

Auto Loader te ofrece:

- Inferencia y evolución de esquema
- Escalabilidad manejando millones de archivos
- Simplicidad: solo define tu carpeta de ingesta; Databricks se encarga del resto

Para más detalles sobre Auto Loader, ejecuta `dbdemos.install('data-ingestion')`

Usemos esto en nuestro pipeline e ingiramos los datos en bruto JSON y CSV entregados en nuestro blob storage `/demos/manufacturing/iot_turbine/...`. 

Abre el cuaderno [transformations/01-bronze.sql]($./transformations/01-bronze.sql) para revisar las consultas SQL que ingieren los datos en bruto y crean nuestra capa bronze.


## 3/ Capa Silver

<div><img src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-2.png" width="700px" style="float: right"/></div>

Para poder analizar nuestros datos, calcularemos métricas estadísticas de forma horaria, como desviación estándar y cuantiles.

*Nota: recomputaremos toda la tabla para mantener este ejemplo simple. En su lugar podríamos hacer UPSERT de la hora actual con una agregación con estado.*

Abre el cuaderno [transformations/02-silver.sql]($./transformations/02-silver.sql) para revisar las consultas SQL que crean nuestras características y nuestro conjunto de entrenamiento


## 4/ Capa Gold: obtén el modelo del registro y marca turbinas defectuosas

<div><img src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-turbine-4.png" width="700px" style="float: right"/></div>

Nuestro equipo de ciencia de datos pudo leer datos de las tablas anteriores y construir un modelo de mantenimiento predictivo usando AutoML, guardándolo en el Registro de Modelos de Databricks (veremos cómo hacerlo a continuación).

Uno de los valores clave del Lakehouse es que podemos cargar fácilmente este modelo y predecir turbinas defectuosas directamente dentro de nuestro pipeline.

Ten en cuenta que no tenemos que preocuparnos por el framework del modelo (sklearn u otro), MLflow lo abstrae por nosotros.

Todo lo que tenemos que hacer es cargar el modelo y llamarlo como una función SQL (o desde Python).

Abre el cuaderno [transformations/03-gold.sql]($./transformations/03-gold.sql) para revisar las consultas SQL que crean nuestras características y nuestro conjunto de entrenamiento


## Conclusión
Nuestro <a dbdemos-pipeline-id="sdp-sql" href="#joblist/pipelines/4d2b04f6-270a-49da-9980-96f393f6ef8c" target="_blank">Spark Declarative Pipeline</a> ya está listo usando únicamente SQL. Tenemos un ciclo de extremo a extremo, y nuestro modelo de ML ha sido integrado sin fricción por el equipo de Ingeniería de Datos.


Para más detalles sobre el entrenamiento del modelo, abre el [notebook de entrenamiento]($../../04-Data-Science-ML/04.1-automl-iot-turbine-predictive-maintenance)

Nuestro dataset final incluye la predicción de ML para el caso de uso de Mantenimiento Predictivo. 

Ahora estamos listos para construir nuestro <a dbdemos-dashboard-id="turbine-analysis" href="/sql/dashboardsv3/01f0b6a83d751bcb973f7564d17676fd">tablero AI/BI</a> para seguir los principales KPIs y el estado de todo nuestro parque eólico, y un <a dbdemos-dashboard-id="turbine-predictive" href="/sql/dashboardsv3/01f0b6a83d751bcb973f7564d17676fd">tablero AI/BI de mantenimiento predictivo</a>.


<img src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/manufacturing/lakehouse-iot-turbine/lakehouse-manuf-iot-dashboard-1.png" width="1000px">